<a href="https://colab.research.google.com/github/hrley55/BIG-DATA-NHOM-7/blob/main/FP_Growth_(mlxtend).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cài đặt thư viện nếu chưa có
!pip install mlxtend -q

# =====================================================================
# TẮT TOÀN BỘ CẢNH BÁO RÁC TỪ COLAB (Phải đặt ở trên cùng)
# =====================================================================
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

# =====================================================================
# 1. XÂY DỰNG TRANSACTION DATASET (ONE-HOT ENCODING)
# =====================================================================
print("\n1. Đang xây dựng transaction dataset...")

# Cập nhật đường dẫn tới thư mục chứa data của bạn
data_path = '/content/drive/MyDrive/Olist_Data/'

# Đọc dữ liệu
df_items = pd.read_csv(f'{data_path}olist_order_items_dataset.csv')
df_products = pd.read_csv(f'{data_path}olist_products_dataset.csv')

# Gộp bảng để lấy product_category_name
df_basket = pd.merge(df_items, df_products, on='product_id', how='inner')
df_basket = df_basket.dropna(subset=['product_category_name'])

# Tạo danh sách các ngành hàng trong từng đơn hàng (order_id)
transactions = df_basket.groupby('order_id')['product_category_name'].apply(list).values.tolist()
print(f"-> Tổng số giao dịch (đơn hàng hợp lệ): {len(transactions)}")

# Thực hiện One-hot encoding bằng TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_transactions = pd.DataFrame(te_ary, columns=te.columns_)
print("-> Đã hoàn thành One-hot encoding. Kích thước ma trận:", df_transactions.shape)


# =====================================================================
# 2. FPGROWTH + ASSOCIATION_RULES (MLXTEND) + PHÂN TÍCH
# =====================================================================
print("\n2. Đang chạy fpgrowth và association_rules...")

# Áp dụng fpgrowth để tìm tập phổ biến
frequent_itemsets = fpgrowth(df_transactions, min_support=0.001, use_colnames=True)

if not frequent_itemsets.empty:
    # Áp dụng association_rules để tìm luật kết hợp
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.05)

    # Sắp xếp theo lift để chuẩn bị cho phần phân tích
    top_10_rules = rules.sort_values(by='lift', ascending=False).head(10)

    print("\n--- KẾT QUẢ TOP 10 LUẬT KẾT HỢP (DÙNG ĐỂ PHÂN TÍCH) ---")
    display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
    display(top_10_rules[display_cols].reset_index(drop=True))

else:
    print("\n[PHÂN TÍCH] Không tìm thấy tập phổ biến nào với min_support = 0.001.")
    print("-> Hướng xử lý: Chuyển sang Tuần 3 tiến hành gom nhóm danh mục (Macro-categories) hoặc giảm min_support.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

1. Đang xây dựng transaction dataset...
-> Tổng số giao dịch (đơn hàng hợp lệ): 97277
-> Đã hoàn thành One-hot encoding. Kích thước ma trận: (97277, 73)

2. Đang chạy fpgrowth và association_rules...

--- KẾT QUẢ TOP 10 LUẬT KẾT HỢP (DÙNG ĐỂ PHÂN TÍCH) ---


,antecedents,consequents,support,confidence,lift
